<a href="https://colab.research.google.com/github/Artur569/ideahub-connect-insight/blob/main/AFD2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from dataclasses import dataclass, field


class AFDInvalido(Exception):
    """A quíntupla fornecida não satisfaz a definição de AFD."""


@dataclass(frozen=True)
class AFD:
    estados: frozenset[str]                    # Q
    alfabeto: frozenset[str]                   # Σ
    transicoes: dict[tuple[str, str], str]     # δ
    inicial: str                               # q₀
    finais: frozenset[str]                     # F

    def __post_init__(self) -> None:
        self.validar()

    def validar(self) -> None:
        if not self.estados:
            raise AFDInvalido("Q não pode ser vazio.")
        if not self.alfabeto:
            raise AFDInvalido("Σ não pode ser vazio.")
        if self.inicial not in self.estados:
            raise AFDInvalido(f"q₀={self.inicial!r} não pertence a Q.")

        fora = self.finais - self.estados
        if fora:
            raise AFDInvalido(f"F contém estados fora de Q: {sorted(fora)}")

        faltando = [
            (q, a)
            for q in sorted(self.estados)
            for a in sorted(self.alfabeto)
            if (q, a) not in self.transicoes
        ]
        if faltando:
            raise AFDInvalido(
                f"δ não é total. Faltam {len(faltando)} transições: {faltando}"
            )

        for (q, a), destino in self.transicoes.items():
            if q not in self.estados:
                raise AFDInvalido(f"δ parte de {q!r}, que não está em Q.")
            if a not in self.alfabeto:
                raise AFDInvalido(f"δ lê {a!r}, que não está em Σ.")
            if destino not in self.estados:
                raise AFDInvalido(f"δ({q},{a}) leva a {destino!r}, fora de Q.")

    def passo(self, estado: str, simbolo: str) -> str:
        """Um passo de δ. Erro claro se o símbolo não pertence a Σ."""
        if simbolo not in self.alfabeto:
            raise ValueError(f"símbolo {simbolo!r} não pertence a Σ={sorted(self.alfabeto)}")
        return self.transicoes[(estado, simbolo)]

    def delta_estendida(self, cadeia: str) -> str:
        """δ̂(q₀, w) — o estado onde a computação termina."""
        estado = self.inicial
        for simbolo in cadeia:
            estado = self.passo(estado, simbolo)
        return estado

    def aceita(self, cadeia: str) -> bool:
        """w ∈ L(M)?"""
        return self.delta_estendida(cadeia) in self.finais

    def trace(self, cadeia: str) -> list[tuple[str, str, str]]:
        """A computação completa, como lista de (antes, símbolo, depois)."""
        passos: list[tuple[str, str, str]] = []
        estado = self.inicial
        for simbolo in cadeia:
            proximo = self.passo(estado, simbolo)
            passos.append((estado, simbolo, proximo))
            estado = proximo
        return passos

def formatar_trace(m: AFD, cadeia: str) -> str:
    linhas = [f"entrada: {cadeia!r}", f"início:  {m.inicial}"]
    estado = m.inicial
    for i, (antes, simbolo, depois) in enumerate(m.trace(cadeia), start=1):
        linhas.append(f"  passo {i}: δ({antes}, {simbolo}) = {depois}")
        estado = depois
    veredito = "ACEITA" if estado in m.finais else "REJEITA"
    linhas.append(f"fim:     {estado}  ({'in' if estado in m.finais else 'not in'} F) -> {veredito}")
    return "\n".join(linhas)

TERMINA_EM_1 = AFD(
    estados=frozenset({"q0", "q1"}),
    alfabeto=frozenset({"0", "1"}),
    transicoes={
        ("q0", "0"): "q0",
        ("q0", "1"): "q1",
        ("q1", "0"): "q0",
        ("q1", "1"): "q1",
    },
    inicial="q0",
    finais=frozenset({"q1"}),
)

print(formatar_trace(TERMINA_EM_1, "1011"))


entrada: '1011'
início:  q0
  passo 1: δ(q0, 1) = q1
  passo 2: δ(q1, 0) = q0
  passo 3: δ(q0, 1) = q1
  passo 4: δ(q1, 1) = q1
fim:     q1  (in F) -> ACEITA
